# Use logs to understand a run

Experiment metrics describe learning. Runtime logs describe execution: which task ran, where a failure occurred, and whether clients responded.

Complete [setup](../01.1_running_federated_learning_job/setup.ipynb). In this section's `code` directory, choose the console detail for a recipe run:

```bash
python job.py --log_config progress
```

| Mode | When to use it |
| --- | --- |
| `concise` | A compact view of the run |
| `progress` | Follow round progress while developing |
| `msg_only` | See message text with less formatting |
| `full` | See detailed formatted log records |
| `verbose` | Investigate with more detailed logging |

The console is only one view. Inspect server and client log files in the workspace for detail. These can contain multiple runs if you reuse a workspace, so use a fresh workspace root when comparing experiments.

In [ ]:
from pathlib import Path
from IPython.display import Code, display

lesson = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "logging.ipynb").is_file())
code = lesson / "code"
assert (code / "job.py").is_file(), f"Keep this notebook beside its code directory: {code}"
print("Run the terminal commands from:", code)

In [ ]:
import os
from pathlib import Path

simulation_root = Path(os.environ.get("NVFLARE_SIMULATOR_WORKSPACE_ROOT", "/tmp/nvflare/simulation"))
workspace = simulation_root / "chapter1-logging"
print(workspace)

In [ ]:
for site in ("server", "site-1", "site-2"):
    folder = workspace / site
    print(site, "logs:")
    for path in sorted(folder.rglob("*log*.txt")):
        print(" ", path)

## Diagnose a failure

1. Read the failure message and note its workspace path.
2. Open the affected site's `error_log.txt` and nearby lines in `log.txt`.
3. Look for the earliest relevant exception, such as an import error, invalid script argument, or model-shape mismatch. Later timeout messages may be consequences.
4. Fix the source script or job definition and rerun in a fresh workspace.

If a run remains waiting, inspect both the server and client logs: the server may be waiting for a client that failed before returning a task result. A result directory by itself is not proof of successful training; also check completion and the expected artifacts.

## Add application messages

Inside an `FLComponent`, use `self.log_info(fl_ctx, "message")` or the controller's `self.info("message")` so the record includes the FL context.

For ordinary Python modules, a standard logger works:

```python
import logging

logger = logging.getLogger(__name__)
logger.info("Starting local training")
```

This section's `client.py` uses `logger.info()` for its settings and round results; those messages appear in the site logs. Keep data samples and credentials out of logs.

## Use a custom logging configuration

A Python `logging.config.dictConfig` JSON file can control handlers, formatters, and levels. Pass its path to the execution environment:

```python
from nvflare.recipe import SimEnv

env = SimEnv(num_clients=2, log_config="/absolute/path/to/log_config.json")
# run = recipe.execute(env)
```

**Try it:** compare `progress` and `full` in separate runs. Find the effective batch size for each client in the log files, then locate the final-evaluation task.

Next: [recap](../01.8_recap/recap.ipynb).